# VLM-Anomaly — Full MVTec Sweep · Claude Sonnet 4.6 via CLI

**Model:** `claude-sonnet-4-6`  
**Backend:** Local `claude` CLI (uses your existing Claude subscription — no separate API billing)  
**Cost estimate:** $0 direct API cost (billed through CLI subscription)  
**Runs locally** against `data/mvtec` dataset.  

> Re-running any cell is safe — the skip check reads `model_id` from file content,
> so only missing categories are processed. Other models' results are never touched.

In [1]:
# ── Cell 1: Setup paths & sys.path ─────────────────────────────────────────
import sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT   = Path().resolve().parent   # notebooks/ → repo root
SRC_DIR     = REPO_ROOT / 'src'
PROMPTS_DIR = REPO_ROOT / 'prompts'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SRC_DIR.exists(),     f'src/ not found at {SRC_DIR}'
assert PROMPTS_DIR.exists(), f'prompts/ not found at {PROMPTS_DIR}'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

import vlm_anomaly
print(f'vlm_anomaly {vlm_anomaly.__version__} ready')
print(f'SRC      : {SRC_DIR}')
print(f'Prompts  : {PROMPTS_DIR}')
print(f'Results  : {RESULTS_DIR}')
from vlm_anomaly.logging import configure_logging
configure_logging(log_level='DEBUG')


vlm_anomaly 0.1.0 ready
SRC      : /Users/sabareeswarans/Projects_26/VLM-Anomaly/src
Prompts  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/prompts
Results  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/results


In [2]:
# ── Cell 2: Verify claude CLI is available ──────────────────────────────────
import shutil, subprocess, json as _json

cli = shutil.which('claude')
assert cli, "'claude' CLI not found on PATH. Run: npm install -g @anthropic-ai/claude-code"

ver = subprocess.run([cli, '--version'], capture_output=True, text=True, timeout=10)
print(f'claude CLI : {cli}')
print(f'version    : {ver.stdout.strip() or ver.stderr.strip()}')
print()
print('Auth check ...')
check = subprocess.run(
    [cli, '--print', '--output-format', 'json', '--model', 'claude-sonnet-4-6',
     '--no-session-persistence',
     '--system-prompt', 'You are a helpful assistant. Reply concisely.',
     'Reply with only the word: ready'],
    capture_output=True, text=True, timeout=60
)
try:
    env = _json.loads(check.stdout)
    result_text = env.get('result', '')
    if env.get('is_error') and 'logged in' in result_text.lower():
        raise RuntimeError('claude CLI not authenticated — run `claude auth login` in your terminal')
    print(f'CLI auth OK — model: claude-sonnet-4-6  response: {result_text[:40]!r}')
    print(f'cost_usd   : ${env.get("total_cost_usd", 0):.5f}')
except _json.JSONDecodeError:
    print(f'CLI stdout: {check.stdout[:200]}')
    raise RuntimeError('Unexpected CLI output — check `claude --version` in your terminal')


claude CLI : /Users/sabareeswarans/.local/bin/claude
version    : 2.1.150 (Claude Code)

Auth check ...
CLI auth OK — model: claude-sonnet-4-6  response: 'Credit balance is too low'
cost_usd   : $0.00000


In [3]:
# ── Cell 3: Find MVTec dataset ──────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
]:
    if candidate.exists() and any(candidate.iterdir()):
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, f'MVTec not found. Expected at {REPO_ROOT}/data/mvtec'
categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {len(categories)} → {categories}')

MVTec root : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Categories : 15 → ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [4]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
MODEL      = 'claude-sonnet-4-6'        # full model ID passed to --model
MODEL_ID   = f'claude_cli/{MODEL}'       # model_id written into JSONL results
PROMPT_KEY = 'manufacturing.detailed'
LIMIT      = None                        # None = all images; set e.g. 5 for a quick test
BUDGET_USD = 50.0                        # safety cap (CLI cost is $0 from API perspective)

print(f'Model      : {MODEL}')
print(f'Model ID   : {MODEL_ID}')
print(f'Prompt     : {PROMPT_KEY}')
print(f'Limit      : {LIMIT or "all images"}')
print(f'Total      : ~{len(categories) * 83} images')

Model      : claude-sonnet-4-6
Model ID   : claude_cli/claude-sonnet-4-6
Prompt     : manufacturing.detailed
Limit      : all images
Total      : ~1245 images


In [5]:
# ── Cell 5: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.backends.claude_cli import ClaudeCliBackend
from vlm_anomaly.evaluators.prompt_library import PromptLibrary
from vlm_anomaly.logging import configure_logging

configure_logging(json_logs=False, log_level='INFO')

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
    default_budget_usd=BUDGET_USD,
)
settings.results_dir = RESULTS_DIR

dataset    = MVTec(root_dir=MVTEC_ROOT)
backend    = ClaudeCliBackend(model=MODEL)
prompt_lib = PromptLibrary(prompts_dir=PROMPTS_DIR)

print(f'Dataset  : {MVTEC_ROOT}')
print(f'Backend  : {backend.name} / {MODEL}')
print(f'Prompts  : {len(list(prompt_lib._prompts.keys()))} keys loaded')

Dataset  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Backend  : claude_cli / claude-sonnet-4-6
Prompts  : 4 keys loaded


In [8]:
# ── Cell 5b: Clean up incomplete / auth-broken claude_cli result files ──────
# Re-run this whenever you need to force a category to be re-evaluated.
# Only deletes files whose model_id matches MODEL_ID (leaves Gemini/Qwen alone).
import json as _json

CLEAN_CATEGORIES = ['bottle', 'cable', 'capsule', 'carpet']  # extend as needed

deleted = []
for cat in CLEAN_CATEGORIES:
    for fpath in RESULTS_DIR.glob(f'*_mvtec_{cat}.jsonl'):
        if fpath.stat().st_size == 0:
            fpath.unlink()
            deleted.append(fpath.name)
            continue
        try:
            first = _json.loads(fpath.read_text().splitlines()[0])
        except Exception:
            fpath.unlink()
            deleted.append(fpath.name)
            continue
        if first.get('model_id') == MODEL_ID:
            fpath.unlink()
            deleted.append(fpath.name)

if deleted:
    print('Deleted (will be re-run by Cell 6 / 6b):')
    for n in deleted:
        print(f'  {n}')
else:
    print('Nothing to clean — no matching claude_cli files found.')


Deleted (will be re-run by Cell 6 / 6b):
  cc3d9b47_mvtec_bottle.jsonl


In [7]:
# ── Cell 6: Run all 15 categories ───────────────────────────────────────────
# Skip check reads model_id from JSONL content — safe to re-run.
import json as _json
from tqdm.auto import tqdm
from vlm_anomaly.schemas import ExperimentConfig
from vlm_anomaly.evaluators.vlm_evaluator import VLMEvaluator


def _already_done(results_dir, category, model_id):
    """Return True if a complete result file for this model+category exists."""
    for f in results_dir.glob(f'*_mvtec_{category}.jsonl'):
        if f.stat().st_size < 100:
            continue
        try:
            rec = _json.loads(f.read_text().splitlines()[0])
            if rec.get('model_id') == model_id:
                return f
        except Exception:
            pass
    return None


all_results = []

for category in tqdm(categories, desc='MVTec categories'):
    done_file = _already_done(RESULTS_DIR, category, MODEL_ID)
    if done_file:
        print(f'  [skip] {category} — already done ({done_file.name})')
        continue

    config = ExperimentConfig(
        backend=MODEL_ID,
        dataset='mvtec',
        categories=[category],
        prompt=PROMPT_KEY,
        limit=LIMIT,
        budget_usd=BUDGET_USD,
    )
    evaluator = VLMEvaluator(
        backend=backend,
        dataset=dataset,
        config=config,
        settings=settings,
        prompt_library=prompt_lib,
    )
    results = evaluator.run()
    all_results.extend(results)
    for r in results:
        print(f'  {category:12s}  AUROC={r.auroc:.3f}  F1={r.f1:.3f}  n={r.n_images}')

print(f'\nDone. {len(all_results)} new categories processed.')

MVTec categories:   0%|          | 0/15 [00:00<?, ?it/s]

2026-05-24T02:07:12.569544Z [info     ] evaluator.run.start            [vlm_anomaly.evaluators.vlm_evaluator] backend=claude_cli budget_usd=50.0 categories=['bottle'] dataset=mvtec experiment_id=cc3d9b47 limit=None prompt=manufacturing.detailed


KeyboardInterrupt: 

In [ ]:
# ── Cell 6b: Run ONLY missing categories ────────────────────────────────────
# Detects which categories this model hasn't finished yet and runs only those.
import json as _json
from tqdm.auto import tqdm
from vlm_anomaly.schemas import ExperimentConfig
from vlm_anomaly.evaluators.vlm_evaluator import VLMEvaluator


def _done_categories(results_dir, model_id):
    done = set()
    for f in results_dir.glob('*_mvtec_*.jsonl'):
        if f.stat().st_size < 100:
            continue
        try:
            rec = _json.loads(f.read_text().splitlines()[0])
            if rec.get('model_id') == model_id:
                done.add(rec.get('category', ''))
        except Exception:
            pass
    return done


done_cats = _done_categories(RESULTS_DIR, MODEL_ID)
missing   = sorted(set(categories) - done_cats)

print(f'Already done : {sorted(done_cats)}')
print(f'Missing      : {missing}')

if not missing:
    print('All 15 categories complete — nothing to run.')
else:
    missing_results = []
    for category in tqdm(missing, desc='Missing categories'):
        config = ExperimentConfig(
            backend=MODEL_ID,
            dataset='mvtec',
            categories=[category],
            prompt=PROMPT_KEY,
            limit=LIMIT,
            budget_usd=BUDGET_USD,
        )
        evaluator = VLMEvaluator(
            backend=backend,
            dataset=dataset,
            config=config,
            settings=settings,
            prompt_library=prompt_lib,
        )
        results = evaluator.run()
        missing_results.extend(results)
        for r in results:
            print(f'  {category:12s}  AUROC={r.auroc:.3f}  F1={r.f1:.3f}  n={r.n_images}')
    print(f'\nDone. {len(missing_results)} missing categories filled in.')

In [ ]:
# ── Cell 7: Leaderboard ─────────────────────────────────────────────────────
import importlib
import vlm_anomaly.analysis.aggregator as _agg_mod
importlib.reload(_agg_mod)
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table

lb = leaderboard(RESULTS_DIR)
if lb.empty:
    print('No results yet.')
else:
    summary = cost_accuracy_table(RESULTS_DIR)
    print('=== Summary — mean AUROC across categories (all models) ===')
    print(summary[['model_id', 'mean_auroc', 'mean_latency_ms']].to_string(index=False))
    print()
    print(f'=== Per-category breakdown — {MODEL_ID} ===')
    cli_lb = lb[lb['model_id'] == MODEL_ID].copy()
    if cli_lb.empty:
        print('  No results for this model yet — run Cell 6 first.')
    else:
        display(
            cli_lb[['category', 'n_images', 'auroc', 'f1', 'mean_latency_ms']]
            .sort_values('auroc', ascending=False)
            .reset_index(drop=True)
        )

In [ ]:
# ── Cell 8: Generate report ──────────────────────────────────────────────────
# Regenerates REPORT.md from ALL models' results.
# Only this model's rows change when this notebook added new results.
import importlib
import vlm_anomaly.analysis.report_generator as _rg_mod
importlib.reload(_rg_mod)
from vlm_anomaly.analysis.report_generator import generate

report = generate(RESULTS_DIR, str(REPO_ROOT / 'REPORT.md'))
print(f'Report written to {report}')

In [ ]:
# ── Cell 9: Show result files + commit hint ──────────────────────────────────
import json as _json

result_files = [
    f for f in sorted(RESULTS_DIR.glob('*_mvtec_*.jsonl'))
    if f.stat().st_size > 100 and
    _json.loads(f.read_text().splitlines()[0]).get('model_id') == MODEL_ID
]
print(f'Claude CLI result files ({len(result_files)}):')  
for f in result_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}  ({size_kb:.1f} KB)')

print()
print('To commit results:')
print('  git add results/*.jsonl REPORT.md')
print(f'  git commit -m "results(claude-cli): full MVTec sweep via CLI backend"')